# 01. Initialize Phoenix Tracing

## Why are we implementing this?

Phoenix provides observability for our SEVA_AI retrieval pipeline.

We use OpenTelemetry and OpenInference to capture execution traces from LangChain components.

This allows us to inspect:
- Retrieval execution
- Retrieved documents
- Execution latency
- Nested operations
- Errors and failures

We will send traces to our locally running Phoenix server at http://localhost:6006.

Our Phoenix project name will be SEVA_AI.

In [1]:
import phoenix.otel
import openinference.instrumentation.langchain

print("Phoenix instrumentation installed successfully!")

Phoenix instrumentation installed successfully!


In [2]:
from phoenix.otel import register
from opentelemetry import trace

tracer_provider = register(
    project_name="SEVA_AI",
    endpoint="http://localhost:6006/v1/traces",
    protocol="http/protobuf",
    batch=True,
    auto_instrument=True
)

tracer = trace.get_tracer("seva_ai.test")

with tracer.start_as_current_span("phoenix_connection_test") as span:
    span.set_attribute("project", "SEVA_AI")
    span.set_attribute("operation", "connection_test")
    span.set_attribute("status", "success")

tracer_provider.force_flush()

print("Phoenix trace submitted.")

OpenTelemetry Tracing Details
|  Phoenix Project: SEVA_AI
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix trace submitted.


## 02. Create First Test Trace

### Why are we implementing this?

We want to verify end-to-end communication between our Python application and Phoenix.

We will manually create a test span using the OpenTelemetry API and export it to our SEVA_AI Phoenix project.

This confirms that our tracing configuration works before instrumenting the actual retrieval pipeline.

In [3]:
from opentelemetry import trace

tracer = trace.get_tracer("seva_ai.test")

with tracer.start_as_current_span("phoenix_connection_test") as span:
    span.set_attribute("project", "SEVA_AI")
    span.set_attribute("operation", "connection_test")
    span.set_attribute("status", "success")

print("Test trace exported to Phoenix.")

Test trace exported to Phoenix.


## 03. Chroma Discovery Retrieval Tracing

### Why are we implementing this?

We want to observe the actual SEVA_AI discovery retrieval operation inside Phoenix.

We will capture:

- User query
- Retrieved chunks
- Scheme IDs
- Similarity distances
- Number of retrieved chunks
- Retrieval execution time
- Retrieval errors

This will help us investigate why certain schemes are retrieved or missed.

We will preserve our existing Chroma collection and retrieval configuration.

In [ ]:
import json
import time

from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode

tracer = trace.get_tracer("seva_ai.discovery")


def traced_discovery_retrieval(query: str, chunk_k: int = 30):

    with tracer.start_as_current_span(
        "seva_discovery_retrieval"
    ) as span:

        span.set_attribute(
            "openinference.span.kind", "RETRIEVER"
        )
        span.set_attribute("input.value", query)
        span.set_attribute("retrieval.top_k", chunk_k)

        start_time = time.perf_counter()

        try:
            results = discovery_db.similarity_search_with_score(
                query,
                k=chunk_k
            )

            latency_ms = (
                time.perf_counter() - start_time
            ) * 1000

            retrieved_documents = []

            for doc, distance in results:
                retrieved_documents.append({
                    "id": str(doc.metadata.get("parent_id", "")),
                    "content": doc.page_content[:1000],
                    "score": float(distance),
                    "metadata": {
                        "scheme_id": str(
                            doc.metadata.get("scheme_id", "")
                        ),
                        "source_field": str(
                            doc.metadata.get("source_field", "")
                        )
                    }
                })

            span.set_attribute(
                "retrieval.documents",
                json.dumps(retrieved_documents, ensure_ascii=False)
            )

            span.set_attribute(
                "output.value",
                json.dumps({
                    "retrieved_chunks": len(results),
                    "latency_ms": round(latency_ms, 2)
                })
            )

            span.set_attribute("retrieval.latency_ms", latency_ms)

            return results

        except Exception as error:
            span.record_exception(error)
            span.set_status(
                Status(StatusCode.ERROR, str(error))
            )
            raise


print("Discovery retrieval tracing function initialized.")

Discovery retrieval tracing function initialized.


Failed to export spans batch due to timeout, max retries or shutdown.


## 04. Execute Discovery Retrieval

### Why are we implementing this?

We need to execute a real discovery query to generate a Phoenix trace.

We will use one of our existing scheme-aware FAQ queries and retrieve the top 30 chunks.

The retrieved results will be visible in the Phoenix trace details.

## 04. Prepare Scheme-Aware Test Query

### Why are we implementing this?

The golden test JSON contains the original scheme name and FAQ question.

We need to reconstruct the scheme-aware retrieval query because the retrieval_query column was created dynamically in our previous notebook and was not necessarily saved to the JSON file.

We will preserve the original golden dataset.

In [3]:
import json
from pathlib import Path

gd = Path(
    r"G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\final\Final_dataset_splits\test.json"
)

with open(gd, "r", encoding="utf-8") as f:
    golden_df = json.load(f)

test_query = (
    f"For the scheme '{golden_df[0]['scheme_name']}': "
    f"{golden_df[0]['question']}"
)

print("Scheme:", golden_df[0]["scheme_name"])
print("Original question:", golden_df[0]["question"])
print("Retrieval query:", test_query)

Scheme: Dayanand Bandodkar Scheme For Higher Education For Orphans
Original question: Is this scheme limited to certain fields of study or open to all disciplines?
Retrieval query: For the scheme 'Dayanand Bandodkar Scheme For Higher Education For Orphans': Is this scheme limited to certain fields of study or open to all disciplines?


## 05. Execute Discovery Retrieval

### Why are we implementing this?

We want to execute a real Chroma discovery search and send its execution trace to Phoenix.

We will retrieve the top 30 chunks using our scheme-aware query.

In [6]:
import json
import time
from pathlib import Path
from collections import defaultdict

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [2]:
OUTPUT_DIR1 = Path("G:\\SEVA_Ai\\SEVA_Ai\\output\\SEVA_CHILDREN")

VECTORSTORE_DIR = OUTPUT_DIR1 / "vectorstores"

PARENT_PATH = OUTPUT_DIR1 / "parent_documents.jsonl"

print("Parent file exists:", PARENT_PATH.exists())
print("Vectorstore directory exists:", VECTORSTORE_DIR.exists())

Parent file exists: True
Vectorstore directory exists: True


## 01. Download Embedding Model Locally

### Why are we implementing this?

Our project uses sentence-transformers/all-MiniLM-L6-v2.

Currently, model initialization may repeatedly involve Hugging Face cache checks or downloading model files.

We want to maintain one local copy of the model inside our project.

Benefits:
- Avoid repeated model downloads.
- Work without internet after downloading.
- Reuse the same model files across notebooks.
- Maintain consistent embeddings across retrieval experiments.

In [7]:
import os

# Must be configured before importing huggingface_hub
os.environ["HF_HUB_DISABLE_XET"] = "1"

from huggingface_hub import snapshot_download
from pathlib import Path

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

MODEL_PATH = Path(
    r"G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2"
)

MODEL_PATH.mkdir(parents=True, exist_ok=True)

download_path = snapshot_download(
    repo_id=MODEL_NAME,
    local_dir=str(MODEL_PATH),
    max_workers=2
)

print("Model downloaded successfully!")
print("Location:", download_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Model downloaded successfully!
Location: G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2


## 01. Verify Local Embedding Model

### Why are we implementing this?

We have downloaded all-MiniLM-L6-v2 from Hugging Face.

Before loading it, we need to verify that the model weights, tokenizer and configuration files exist in our local directory.

This ensures that we can load the model without downloading files again.

In [4]:
from pathlib import Path

MODEL_PATH = Path(
    r"G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2"
)

required_files = [
    "config.json",
    "model.safetensors",
    "modules.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.txt",
    "1_Pooling/config.json"
]

missing_files = [
    file for file in required_files
    if not (MODEL_PATH / file).exists()
]

if not missing_files:
    print("Model files verified successfully!")
    print("Model location:", MODEL_PATH)
else:
    print("Missing files:", missing_files)

Model files verified successfully!
Model location: G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2


In [5]:
import os
import time

os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings

start_time = time.perf_counter()

embedding_model = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

load_time = time.perf_counter() - start_time

test_embedding = embedding_model.embed_query(
    "What government schemes are available for farmers?"
)

print(f"Model loading time: {load_time:.2f} seconds")
print("Embedding dimensions:", len(test_embedding))
print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loading time: 121.68 seconds
Embedding dimensions: 384
Model loaded successfully!


## 03. Benchmark Local Embedding Performance

### Why are we implementing this?

We want to measure actual embedding generation latency after the model has loaded.

This helps us understand:
- First inference latency
- Repeated inference latency
- Whether the model is being reused correctly

We will not reload the model during this experiment.

In [6]:
import time

test_query = "What government schemes are available for farmers?"

# Warm-up
embedding_model.embed_query(test_query)

# Benchmark repeated inference
latencies = []

for i in range(5):
    start = time.perf_counter()

    embedding = embedding_model.embed_query(test_query)

    latency = (time.perf_counter() - start) * 1000
    latencies.append(latency)

    print(f"Run {i + 1}: {latency:.2f} ms")

print(f"\nAverage latency: {sum(latencies) / len(latencies):.2f} ms")
print(f"Dimensions: {len(embedding)}")

Run 1: 84.37 ms
Run 2: 354.22 ms
Run 3: 147.57 ms
Run 4: 90.13 ms
Run 5: 127.80 ms

Average latency: 160.82 ms
Dimensions: 384


## 04. Reuse Local Embedding Model

### Why are we implementing this?

We want to maintain a single embedding model instance throughout our SEVA_AI experiments.

The Phoenix integration notebook should use the same locally downloaded model configuration as our retrieval experiments.

This avoids unnecessary model initialization and ensures consistent embeddings.

In [14]:
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings

MODEL_PATH = Path(
    r"G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2"
)

embedding_model = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Local embedding model initialized.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Local embedding model initialized.


## 05. Load Existing Chroma Vector Stores

### Why are we implementing this?

We need to connect our existing SEVA_AI vector databases to the Phoenix integration notebook.

We already have two Chroma collections:

1. seva_discovery_v1 — 23,438 chunks
2. seva_detail_v1 — 67,945 chunks

We will load both collections using our locally downloaded embedding model.

This allows us to trace the actual retrieval pipeline without rebuilding the indexes.

We will preserve the existing collection names, embeddings and persistent storage.

In [7]:
from pathlib import Path
from langchain_chroma import Chroma

PROJECT_ROOT = Path(r"G:\SEVA_Ai\SEVA_Ai")

VECTORSTORE_DIR = (
    PROJECT_ROOT
    / "output"
    / "SEVA_CHILDREN"
    / "vectorstores"
)

DISCOVERY_PATH = VECTORSTORE_DIR / "discovery"
DETAIL_PATH = VECTORSTORE_DIR / "detail"

if not DISCOVERY_PATH.exists():
    raise FileNotFoundError(DISCOVERY_PATH)

if not DETAIL_PATH.exists():
    raise FileNotFoundError(DETAIL_PATH)

discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    embedding_function=embedding_model,
    persist_directory=str(DISCOVERY_PATH)
)

detail_db = Chroma(
    collection_name="seva_detail_v1",
    embedding_function=embedding_model,
    persist_directory=str(DETAIL_PATH)
)

print("Discovery collection:", discovery_db._collection.count())
print("Detail collection:", detail_db._collection.count())
print("Chroma vector stores loaded successfully!")

Discovery collection: 23438
Detail collection: 67945
Chroma vector stores loaded successfully!


## 06. Instrument Discovery Retrieval

### Why are we implementing this?

We want Phoenix to capture the actual execution of our Chroma discovery retrieval.

We will create a custom OpenTelemetry span around the existing similarity search operation.

We will capture:
- Input query
- Retrieved chunk content
- Scheme IDs
- Source fields
- Retrieval distances
- Number of retrieved chunks
- Execution latency

This will allow us to inspect retrieval behavior directly in Phoenix without changing our existing retrieval architecture.

In [16]:
import json
import time

from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode

tracer = trace.get_tracer("seva_ai.discovery")


def traced_discovery_retrieval(query: str, chunk_k: int = 30):

    with tracer.start_as_current_span(
        "seva_discovery_retrieval"
    ) as span:

        span.set_attribute("openinference.span.kind", "RETRIEVER")
        span.set_attribute("input.value", query)
        span.set_attribute("retrieval.top_k", chunk_k)

        start_time = time.perf_counter()

        try:
            results = discovery_db.similarity_search_with_score(
                query,
                k=chunk_k
            )

            latency_ms = (
                time.perf_counter() - start_time
            ) * 1000

            retrieved_documents = []

            for index, (doc, distance) in enumerate(results):

                retrieved_documents.append({
                    "id": str(
                        doc.metadata.get(
                            "parent_id",
                            f"chunk_{index}"
                        )
                    ),
                    "content": doc.page_content,
                    "score": float(distance),
                    "metadata": {
                        "scheme_id": str(
                            doc.metadata.get("scheme_id", "")
                        ),
                        "source_field": str(
                            doc.metadata.get("source_field", "")
                        )
                    }
                })

            span.set_attribute(
                "retrieval.documents",
                json.dumps(
                    retrieved_documents,
                    ensure_ascii=False
                )
            )

            span.set_attribute(
                "retrieval.latency_ms",
                latency_ms
            )

            span.set_attribute(
                "retrieval.chunk_count",
                len(results)
            )

            span.set_attribute(
                "output.value",
                json.dumps({
                    "retrieved_chunks": len(results),
                    "latency_ms": round(latency_ms, 2)
                })
            )

            return results

        except Exception as error:
            span.record_exception(error)
            span.set_status(
                Status(StatusCode.ERROR, str(error))
            )
            raise


print("Discovery retrieval instrumentation initialized.")

Discovery retrieval instrumentation initialized.


In [17]:
test_record = golden_df[0]

test_query = (
    f"For the scheme '{test_record['scheme_name']}': "
    f"{test_record['question']}"
)

discovery_results = traced_discovery_retrieval(
    query=test_query,
    chunk_k=30
)

tracer_provider.force_flush()

print("Retrieved chunks:", len(discovery_results))
print("Phoenix trace submitted.")

Transient error HTTPConnectionPool(host='localhost', port=6006): Max retries exceeded with url: /v1/traces (Caused by NewConnectionError("HTTPConnection(host='localhost', port=6006): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it")) encountered while exporting spans batch, retrying in 0.87s.
Failed to export spans batch due to timeout, max retries or shutdown.


Retrieved chunks: 30
Phoenix trace submitted.


In [20]:
discovery_results = traced_discovery_retrieval(
    query=test_query,
    chunk_k=30
)

tracer_provider.force_flush()

print("Retrieved chunks:", len(discovery_results))
print("Phoenix trace exported.")

Retrieved chunks: 30
Phoenix trace exported.
